In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

try:
    names = []
    for m in client.models.list():
        names.append(m.name)

    print("\n".join(names[:50]))
    print("\nTOTAL:", len(names))
except Exception as e:
    print("models.list() failed:", type(e).__name__, e)
    print("Just use a known model like: gemini-2.0-flash or gemini-2.5-pro")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

models = [m.name for m in client.models.list()]
print("\n".join(models[:50]))
print("\nTOTAL:", len(models))

In [ ]:
###1)Basic Gemini call (no agent)
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")  #Most capable, but also most expensive
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")   #Cheapest

result = llm.invoke("Give me a fact about cats")

print(result.content)



content='A group of cats is called a **clowder**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-pro', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019c88d6-d3db-7543-8dd5-18d097113911-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 918, 'total_tokens': 925, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 906}}


In [6]:
###1)Basic Gemini call (no agent)
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")  #Most capable, but also most expensive
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")   #Cheapest

# result = llm.invoke("Give me a funny tweet about today's weather in New York")
result = llm.invoke("Today is what day of the week?")

print(result.content)



Today is **Friday**.


In [ ]:
import langchain
print(langchain.__version__)

1.2.10


In [ ]:
###2) ReAct agent + Tavily search tool
#############This code is deprecated, use the below code instead
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

from langchain.agents import initialize_agent
from langchain_community.tools import TavilySearchResults

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

search_tool = TavilySearchResults(search_depth="basic")
tools = [search_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True
)

agent.invoke("Give me a funny tweet about today's weather in New York")


##Codes are deprecaed, use the below code instead


In [1]:
##########This is working code for ReAct agent + Tavily search tool, with the latest Langchain agent framework (AgentExecutorV2)
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_community.tools import TavilySearchResults
from langchain.agents import create_agent

from langchain_tavily import TavilySearch
search_tool = TavilySearch(search_depth="basic")

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

# search_tool = TavilySearchResults(search_depth="basic")
search_tool = TavilySearch(search_depth="basic")

tools = [search_tool]

agent = create_agent(model=llm, tools=tools)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Give me a funny tweet about today's weather in New York"}]
})

# The agent returns state; last message is the final answer
print(result["messages"][-1].content)

My weather app says "freezing fog" in New York, but I'm pretty sure that's just the vape clouds from a hipster convention that got lost on their way to Brooklyn. #NYCWeather


In [ ]:
##########This is working code for ReAct agent + Tavily search tool, with the latest Langchain agent framework (AgentExecutorV2)
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_community.tools import TavilySearchResults
from langchain.agents import create_agent

from langchain_tavily import TavilySearch
search_tool = TavilySearch(search_depth="basic")

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

# search_tool = TavilySearchResults(search_depth="basic")
search_tool = TavilySearch(search_depth="basic")

tools = [search_tool]

agent = create_agent(model=llm, tools=tools)

result = agent.invoke({
    "messages": [
    {
        "role": "user",
        "content": (
            "Search for the current weather in New York City, NY today. "
            "Use reliable sources. Extract the actual temperature and condition. "
            "Then write ONE funny tweet about New York City weather using that real data. "
            "Do NOT mention any other city."
        )
    }
]
})

# The agent returns state; last message is the final answer
# print(result["messages"][-1].content)
msg = result["messages"][-1].content
print(msg if isinstance(msg, str) else "".join(p.get("text","") for p in msg))




In [7]:
###3) ReAct agent + Tavily + custom “system time” tool
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_tavily import TavilySearch

import datetime

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro")

search_tool = TavilySearch(search_depth="basic")

@tool
def get_system_time(format: str = "%Y-%m-%d %H:%M:%S"):
    """Returns the current date and time in the specified format."""
    current_time = datetime.datetime.now()
    formatted_time = current_time.strftime(format)
    return formatted_time

tools = [search_tool, get_system_time]

# agent = create_agent(
#     tools=tools,
#     model=llm,
#     agent="zero-shot-react-description",
#     verbose=True
# )

agent = create_agent(
    model=llm,
    tools=tools
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Use Tavily to find SpaceX's most recent launch date/time (UTC). "
                "Use get_system_time() to get the current time. "
                "Then calculate how many days ago the launch was from now. "
                "Return a short final answer with the launch name/date and days ago."
            )
        }
    ]
})  

msg = result["messages"][-1].content
print(msg if isinstance(msg, str) else "".join(p.get("text","") for p in msg))

The most recent SpaceX launch was Starlink 6-104 on February 22, 2026 UTC, which was 1 day ago.


In [ ]:
import sys
print(sys.executable)
import langchain
print(langchain.__version__)


In [8]:
import datetime
print(datetime.datetime.now())

2026-02-23 01:14:47.086762
